In [ ]:
import torch
import numpy as np
import torch.nn as nn
from scipy.stats import norm

In [ ]:
# okay what this distribution actually does it
# input => x = (batch,patches,embed_dim) = (32,196,768) => here we are not dealing with cls token
# sampling distribution is => Uniform Distribution
## 25% visible patches and 75% masked patches

In [ ]:
N = 196
B = 32
E = 768
mask_ratio = 0.75
x = torch.zeros(B,N,E)
x.shape

torch.Size([32, 196, 768])

In [ ]:
## only taking the 25 % patches from the single batch
patches_keep = int(N*(1-mask_ratio))
patches_keep

49

In [ ]:
# building and similar distribution
noise = torch.rand(B, N)
ids_shuffle = torch.argsort(noise, dim=1)

In [ ]:
ids_shuffle

tensor([[  9, 180,  91,  ...,  67, 192,  44],
        [114, 129,  36,  ..., 185,  61, 159],
        [181, 163,  85,  ..., 194, 191,  50],
        ...,
        [121, 182, 150,  ...,  61,   0, 193],
        [ 98, 124,  85,  ..., 166,  97, 149],
        [ 82,   3,  51,  ..., 144,  90,  60]])

In [ ]:
ids_restore = torch.argsort(ids_shuffle, dim=1)
ids_restore.shape

torch.Size([32, 196])

In [ ]:
ids_restore

tensor([[ 58, 164,  89,  ..., 162,  84, 134],
        [ 85, 167,  92,  ..., 144,  48, 186],
        [105, 176,   3,  ..., 131, 193,  60],
        ...,
        [194,  91, 125,  ..., 195,  65,  14],
        [134, 188,   5,  ...,  15,  50,  53],
        [ 54, 103, 187,  ...,  88,  97,  29]])

In [ ]:
ids_keep = ids_shuffle[:, :patches_keep]

In [ ]:
x_visible = torch.gather(
        x,
        dim=1,
        index=ids_keep.unsqueeze(-1).repeat(1, 1, E)
    )

In [ ]:
x_visible.shape

torch.Size([32, 49, 768])

## before going in encoder

In [ ]:
## x = (32,196,768)
# battch = 32 and each image has 196 patches
# noise is calulated for each batch each patch - noise(B,N)
## number of patches to keep is calculated - N*(1-mask_ratio)
## sorting at the dimnersion - ids sorting torch.argsort(noise,dim=1) - shuffle
## resorting to store while inverse masking - torch.argsort(shuffle,dim=1)
## taking the patches = shuffle[:,:number_patch] - ids_keep
## taking the x_mask = torch.gather( x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D) )
## creating and mask for the inverse ids
## mask = torch.ones(B,N)
## mask[B,:patches] = 0
## torch.gather(mask,dim=1,index = ids_restore)

In [ ]:
def mask(x,mask_ratio=0.75):
  B,N,D = x.size()
  uniform = torch.rand(B,N)
  len_keep = int(N*(1-mask_ratio))
  ids_shuffle = torch.argsort(uniform,dim=1) # [[[0.4,0.3,0.5]]] -> argsort -> [[[0,1,2]]] -> dim=1 -> [[1,0,2]]
  ids_restore = torch.argsort(ids_shuffle,dim=1) # [[[1,0,2]]] -> argsort , dim=1 -> [[[0,1,2]]] to keep the track which had sorted
  ids_keep = ids_shuffle[:,:len_keep] # only the 25% are considering
  x_masked = torch.gather(x,dim=1,index=ids_keep.unsqueeze(-1).repeat(1,1,D)) # here in the index we are giving the index that er want i.e we add the ebeddding dim
  ## the mask token that we will use while inverse masking
  mask = torch.ones(B,N)
  mask[:,:len_keep] = 0 # setting the len we want to zeros
  mask = torch.gather(mask,dim=1,index=ids_restore)
  return x_masked,mask,ids_restore

In [ ]:
x_,mask,ids = mask(x,0.75)

In [ ]:
x_.shape

torch.Size([32, 49, 768])

In [ ]:
mask.shape

torch.Size([32, 196])

# Before going in decoder

In [32]:
encoder_output = torch.zeros(32,49,768)
masked_patches = 147
masked_tokens = torch.zeros(32,147,768)
tokens = torch.cat([encoder_output,masked_tokens],dim=1) # here the remaning masked patches are concataned at the end then the pathes are correctly taken
x = torch.gather(
    tokens,dim=1,index = ids_restore.unsqueeze(-1).repeat(1,1,768)
)

In [31]:
x.shape

torch.Size([32, 196, 768])